# 04 — Identical-Split Baselines & Controlled Q1 Ablations

Trains controlled ablation variants on the exact same patient splits:
- `A1`: Random Masking (MAE baseline)
- `A2`: Gradient-Only SGM (V1 masking)
- `A3`: Pan-Organ Net V2 without `MT-MoE`
- `A4`: Pan-Organ Net V2 without `SACR`
- `B1` / `B3`: Narrow single-task baselines (`ConvNeXt-Small` classifier, `UNet-2.5D` segmenter).


In [ ]:
import os, sys, shutil
from pathlib import Path
from google.colab import drive

drive.mount('/content/drive', force_remount=False)

DRIVE_ROOT = Path('/content/drive/MyDrive/ResearchPilot_PanOrgan/V2')
CODE_DIR = DRIVE_ROOT / 'code'
SHARDS_DIR = DRIVE_ROOT / 'preprocessed_shards'
CKPT_DIR = DRIVE_ROOT / 'checkpoints'
EXP_DIR = DRIVE_ROOT / 'experiments'
for d in [SHARDS_DIR, CKPT_DIR, EXP_DIR]:
    d.mkdir(parents=True, exist_ok=True)

# Unpack code package onto local fast SSD if uploaded as tarball, or install directly
if (CODE_DIR / 'V2_code_package.tar.gz').exists():
    Path('/content/panorgan_v2').mkdir(parents=True, exist_ok=True)
    !tar -xzf "{CODE_DIR}/V2_code_package.tar.gz" -C /content/panorgan_v2

PKG_ROOT = Path('/content/panorgan_v2') if Path('/content/panorgan_v2/pyproject.toml').exists() else CODE_DIR
if str(PKG_ROOT) not in sys.path:
    sys.path.insert(0, str(PKG_ROOT))
!pip install -q --no-deps -e "{PKG_ROOT}"
!pip install -q timm nibabel pydicom pyarrow scikit-learn scipy PyYAML tqdm open-clip-torch kaggle

import importlib
importlib.invalidate_caches()
import torch
from panorgan.config import load_config, detect_runtime_profile, seed_everything

cfg = load_config(PKG_ROOT / 'configs' / 'v2_panorgan.yaml')
seed_everything(cfg['project']['seed'], cfg['project']['deterministic'])
runtime = detect_runtime_profile(cfg)
print(f'Active Runtime -> GPU: {runtime.gpu_name} | Tier: {runtime.tier} | Batch: {runtime.batch_size}x{runtime.grad_accum} | AMP: {runtime.amp_dtype}')


In [ ]:
import copy, json
import pandas as pd
from torch.utils.data import DataLoader
from panorgan.data import BalancedModalityBatchSampler, PanOrganShardDataset, ShardCacheManager
from panorgan.models import PanOrganNetV2
from panorgan.engine import ColabResilientTrainer

manifest_df = pd.read_parquet(SHARDS_DIR / 'master_manifest.parquet')
train_df = manifest_df[manifest_df['split'] == 'train'].reset_index(drop=True)
val_df = manifest_df[manifest_df['split'] == 'val'].reset_index(drop=True)
mod_map = {k: int(v['id']) for k, v in cfg['modalities'].items()}
cache_mgr = ShardCacheManager(SHARDS_DIR, Path(cfg['storage']['local_cache_dir']), float(cfg['storage']['max_local_cache_gb']))
cache_mgr.warmup_all_shards()

train_ds = PanOrganShardDataset(train_df, cache_mgr, mod_map, target_hw=(256, 256), augment=True, mnar_truncation=0.20)
val_ds = PanOrganShardDataset(val_df, cache_mgr, mod_map, target_hw=(256, 256), augment=False)
sampler = BalancedModalityBatchSampler(train_df['dataset'].tolist(), batch_size=runtime.batch_size)
train_loader = DataLoader(train_ds, batch_sampler=sampler, num_workers=cfg['storage']['num_workers'], pin_memory=True)
val_loader = DataLoader(val_ds, batch_size=runtime.batch_size, shuffle=False, num_workers=cfg['storage']['num_workers'], pin_memory=True)

ABLATIONS = {
    'ablation_A1_random_mask': {'masking_mode': 'random', 'use_moe': True, 'use_sacr': True},
    'ablation_A2_grad_sgm': {'masking_mode': 'gradient_sgm', 'use_moe': True, 'use_sacr': True},
    'ablation_A3_no_moe': {'masking_mode': 'ma_ugm', 'use_moe': False, 'use_sacr': True},
    'ablation_A4_no_sacr': {'masking_mode': 'ma_ugm', 'use_moe': True, 'use_sacr': False},
}

summary = {}
for exp_name, overrides in ABLATIONS.items():
    print(f'\n=== Running {exp_name} ===')
    ab_cfg = copy.deepcopy(cfg)
    ab_cfg['model'].update(overrides)
    model = PanOrganNetV2.from_config(ab_cfg)
    trainer = ColabResilientTrainer(
        model=model,
        runtime=runtime,
        checkpoint_dir=CKPT_DIR / exp_name,
        lr=float(cfg['training']['stage2_multitask']['lr']),
        patience=5,
    )
    trainer.fit(train_loader, val_loader, num_epochs=15, mode='multitask', resume=True)
    summary[exp_name] = {'best_val_loss': trainer.best_val_loss}

(EXP_DIR / 'ablation_training_summary.json').write_text(json.dumps(summary, indent=2), encoding='utf-8')
print(json.dumps(summary, indent=2))
